# Lekcija 29: Struktura iz gibanja

Ova lekcija povezuje podudaranje značajki (lekcija 20), robusnu procjenu (lekcija 23), kalibraciju (lekcija 27) te esencijalnu matricu i određivanje položaja i orijentacije kamere (lekcija 28) u cjelovit postupak koji uzima 2D korespondencije iz dvaju pogleda i određuje **i** relativno gibanje kamera **i** 3D položaje promatranih točaka &mdash; **S**tructure **f**rom **M**otion, odnosno strukturu iz gibanja. Jedini sastojak koji još nismo detaljno razradili jest **triangulacija**: pretvaranje para odgovarajućih 2D točaka, uz poznate položaje i orijentacije kamera, u 3D točku. (Lekcija 22 pokazala je kako disparitet pretvoriti u dubinu za rektificirane slike; ova lekcija poopćuje tu ideju.)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Sažeti klasični postupak

1. **Podudariti značajke** između dviju slika (lekcija 20: SIFT + test omjera).
2. **Procijeniti esencijalnu matricu** $E$ iz tih korespondencija, robusno (lekcija 23: RANSAC), uz poznate unutarnje parametre $K$ (lekcija 27: kalibracija).
3. **Odrediti relativni položaj i orijentaciju** $(R, t)$ iz $E$ (lekcija 28: `cv2.recoverPose`) &mdash; do nepoznatog faktora mjerila za $t$.
4. **Triangulirati**: za svaki podudareni par presjeći dvije odgovarajuće zrake u 3D prostoru kako bi se odredila 3D točka &mdash; to se prvi put detaljno razvija u ovoj lekciji.

Koristimo dvije realistične sintetičke fotografije brončane skulpture iz skupa podataka BlendedMVS (<a href="../references.html#yao-2020-blendedmvs">Yao i sur., 2020.</a>). Za razliku od para fotografija koji bismo sami snimili, ove slike dolaze s poznatim stvarnim položajima i orijentacijama kamera, kalibracijom i geometrijom za provjeru cijelog postupka.

## Dvije stvarne fotografije s poznatim položajima i orijentacijama kamera

Kao što je spomenuto, svaka slika dolazi s poznatom stvarnom vrijednošću $(K, R, t)$. Kameru prve fotografije uzimamo kao ishodište svijeta, a položaj i orijentaciju druge izražavamo u odnosu na nju &mdash; tako dobivamo `P1` i `P2_true`, dvije stvarne projekcijske matrice koje cijela ova lekcija pokušava odrediti samo iz korespondencija na slikama.

In [ ]:
K = np.array([[583.2225, 0., 255.4034],
              [0., 583.2225, 188.8833],
              [0., 0., 1.]])

R1, t1 = np.eye(3), np.zeros(3)
R_true = np.array([[ 0.92083152, -0.14714329,  0.36113446],
                    [ 0.12698365,  0.98874714,  0.07907683],
                    [-0.36870643, -0.02695794,  0.92915445]])
t_true = np.array([-0.78751177, 0.00981936, 0.03364542])

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2_true = K @ np.hstack([R_true, t_true.reshape(3, 1)])

view1 = cv2.imread('../img/bronzebull00.jpg')
view2 = cv2.imread('../img/bronzebull02.jpg')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(view1, cv2.COLOR_BGR2RGB))
axes[0].set_title('View 1')
axes[1].imshow(cv2.cvtColor(view2, cv2.COLOR_BGR2RGB))
axes[1].set_title('View 2')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: <a href="https://github.com/YoYo000/BlendedMVS" target="_blank" rel="noopener">BlendedMVS</a> (CC BY 4.0)</p>

## Korak 1: podudaranje značajki

SIFT + test omjera (lekcija 20), primijenjeni izravno na dvije fotografije.

In [ ]:
gray1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

x1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
x2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

print(f'keypoints: {len(kp1)} (view 1), {len(kp2)} (view 2)')
print(f'good matches after ratio test: {len(good_matches)}')

match_vis = cv2.drawMatches(view1, kp1, view2, kp2, good_matches, None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(10, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title('Feature matches')
plt.axis('off')
plt.show()

## Korak 2: robusna procjena esencijalne matrice te položaja i orijentacije kamera

Propustite podudarene točke kroz isti robusni postupak izgrađen u lekcijama 23 i 27 kako biste odredili esencijalnu matricu i relativni položaj i orijentaciju kamera.

In [ ]:
E, inlier_mask = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)
_, R_estimated, t_estimated, _ = cv2.recoverPose(E, x1, x2, K)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered rotation:\n', np.round(R_estimated, 4))
print('true rotation:\n', np.round(R_true, 4))
print()
print('recovered translation direction:', np.round(t_estimated.ravel(), 4))
print('true translation direction:     ', np.round(t_true / np.linalg.norm(t_true), 4))

Kao u lekciji 28, `recoverPose` vraća smjer translacije **jedinične duljine** &mdash; stvarni razmak između kamera načelno se ne može odrediti samo iz korespondencija na slikama. Na to ćemo se vratiti.

## Korak 3: triangulacija

Uz dvije projekcijske matrice kamera $P_1, P_2$ (svaka dimenzija $3\times4$, koja preslikava 3D točku u 2D točku slike u homogenim koordinatama) i par odgovarajućih piksela $(x_1, x_2)$, tražimo 3D točku $X$ koja zadovoljava i $x_1 \propto P_1 X$ i $x_2 \propto P_2 X$. Svaki pogled daje 2 neovisne linearne jednadžbe s 4 homogene nepoznanice točke $X$ (eliminacijom nepoznatog faktora mjerila unakrsnim množenjem), prema istom postupku DLT-a/SVD-a korištenom za homografije (lekcija 24) i fundamentalnu matricu (lekcija 28):

$$A = \begin{bmatrix} u_1 P_1^{(3)} - P_1^{(1)} \\ v_1 P_1^{(3)} - P_1^{(2)} \\ u_2 P_2^{(3)} - P_2^{(1)} \\ v_2 P_2^{(3)} - P_2^{(2)} \end{bmatrix}, \qquad AX = 0$$

gdje $P^{(i)}$ označava redak $i$ matrice $P$. Nul-prostor matrice $A$ (desni singularni vektor pridružen najmanjoj singularnoj vrijednosti) daje $X$.

In [ ]:
def triangulate_dlt(P1, P2, x1, x2):
    points = []
    for (u1, v1), (u2, v2) in zip(x1, x2):
        A = np.array([
            u1 * P1[2] - P1[0],
            v1 * P1[2] - P1[1],
            u2 * P2[2] - P2[0],
            v2 * P2[2] - P2[1],
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        points.append(X[:3] / X[3])
    return np.array(points)

# validate against the dataset's TRUE camera poses: triangulate the RANSAC inliers with them,
# and check that reprojecting the resulting 3D points lands back on the original pixels
inliers = inlier_mask.ravel().astype(bool)
x1_in, x2_in = x1[inliers], x2[inliers]

recon_true_cameras = triangulate_dlt(P1, P2_true, x1_in, x2_in)

cv_points_4d = cv2.triangulatePoints(P1, P2_true, x1_in.T, x2_in.T)
cv_points_3d = (cv_points_4d[:3] / cv_points_4d[3]).T
print(f'max diff vs. cv2.triangulatePoints: {np.abs(recon_true_cameras - cv_points_3d).max():.2e}')

# cheirality check: a correctly-matched 3D point must be in front of BOTH cameras --
# a real pipeline always applies this, since epipolar-only RANSAC can still admit a bad match
depth1 = recon_true_cameras[:, 2]
depth2 = (R_true @ recon_true_cameras.T + t_true.reshape(3, 1))[2]
valid = (depth1 > 0) & (depth2 > 0)
x1_in, x2_in, recon_true_cameras = x1_in[valid], x2_in[valid], recon_true_cameras[valid]
print(f'kept {valid.sum()} / {len(valid)} inliers after cheirality check')

def reprojection_error(P, X, x):
    X_h = np.hstack([X, np.ones((len(X), 1))])
    proj = (P @ X_h.T).T
    proj = proj[:, :2] / proj[:, 2:3]
    return np.linalg.norm(proj - x, axis=1)

err1 = reprojection_error(P1, recon_true_cameras, x1_in)
err2 = reprojection_error(P2_true, recon_true_cameras, x2_in)
all_err = np.concatenate([err1, err2])
print(f'reprojection error using TRUE camera poses: mean {all_err.mean():.3f} px, max {all_err.max():.3f} px')

### Cheiralnost: besplatna provjera smislenosti

Rastavljanje $E$ na $(R, t)$ u koraku 2 zapravo daje četiri matematički valjane kombinacije predznaka; samo jedna smješta promatrane točke ispred *obiju* kamera. `cv2.recoverPose` upravo tu činjenicu &mdash; **ograničenje cheiralnosti** &mdash; interno koristi za odabir ispravne kombinacije, triangulirajući nekoliko točaka i provjeravajući njihovu dubinu. Gornja provjera izričito primjenjuje istu ideju: samo epipolarno ograničenje ne isključuje korespondenciju koja se triangulira *iza* kamere, pa pogrešno podudarena točka još uvijek može proći RANSAC-ov test pripadnosti. Zahtjev pozitivne dubine u oba pogleda besplatan je dodatni filtar &mdash; upravo je on uklonio jednu gore označenu točku koja je prošla RANSAC.

## Povezivanje svega: rekonstrukcija s *procijenjenim* položajem i orijentacijom

Sada slijedi prava provjera: trianguliramo pomoću matrice kamere izgrađene iz *procijenjenih* $(R, t)$ koje daje `recoverPose`, umjesto stvarnog položaja i orijentacije iz skupa podataka. Budući da je $t$ određen samo do faktora mjerila, isto vrijedi i za rekonstrukciju &mdash; svaka 3D točka ispada manja od stvarnosti za isti faktor. Množenje stvarnim razmakom kamera (malo varanja, samo radi evaluacije) trebalo bi dati ispravan metrički prizor.

In [ ]:
P2_estimated = K @ np.hstack([R_estimated, t_estimated.reshape(3, 1)])
reconstruction_unit_scale = triangulate_dlt(P1, P2_estimated, x1_in, x2_in)

true_baseline = np.linalg.norm(t_true)
reconstruction_metric = reconstruction_unit_scale * true_baseline

error = np.linalg.norm(reconstruction_metric - recon_true_cameras, axis=1)
print(f'true baseline: {true_baseline:.3f}')
print(f'mean 3D reconstruction error (vs. triangulation from true poses): {error.mean():.3f}')
print(f'max 3D reconstruction error:                                     {error.max():.3f}')

Uz stvarne (nesavršene) korespondencije, cijeli postupak &mdash; esencijalna matrica, položaj i orijentacija, triangulacija &mdash; rekonstruira prizor s pogreškom koja je mali dio razmaka kamera, koristeći samo 2D pikselne korespondencije i poznate unutarnje parametre, uz *jedan* vanjski broj (stvarni razmak kamera) za uklanjanje neodređenosti mjerila. U praksi ta referenca mjerila može dolaziti iz poznate veličine objekta u prizoru, drugog senzora (GPS, IMU, LiDAR) ili kalibriranog stereo postava (lekcija 22), umjesto iz dviju proizvoljnih neovisnih kamera.

### Vizualizacija rekonstrukcije

Za gušći i prepoznatljiviji oblak točaka od strogog skupa točaka koje su prošle RANSAC, korištenog u gornjoj numeričkoj provjeri, ublažavamo pragove podudaranja značajki (`SIFT_create(contrastThreshold=0.01)`, omjer 0.85) kako bismo uključili više &mdash; nešto bučnijih &mdash; korespondencija, a zatim svaku 3D točku bojimo njezinom stvarnom bojom piksela iz pogleda 1.

In [ ]:
sift_dense = cv2.SIFT_create(contrastThreshold=0.01)
kp1d, des1d = sift_dense.detectAndCompute(gray1, None)
kp2d, des2d = sift_dense.detectAndCompute(gray2, None)
raw_dense = bf.knnMatch(des1d, des2d, k=2)
good_dense = [m for m, n in raw_dense if m.distance < 0.85 * n.distance]

x1d = np.float32([kp1d[m.queryIdx].pt for m in good_dense])
x2d = np.float32([kp2d[m.trainIdx].pt for m in good_dense])

_, dense_mask = cv2.findEssentialMat(x1d, x2d, K, method=cv2.RANSAC, threshold=1.0)
dense_inliers = dense_mask.ravel().astype(bool)
x1d, x2d = x1d[dense_inliers], x2d[dense_inliers]

recon_dense_true = triangulate_dlt(P1, P2_true, x1d, x2d)
recon_dense_estimated = triangulate_dlt(P1, P2_estimated, x1d, x2d) * true_baseline

# cheirality check again, same as for the sparse set above
depth1 = recon_dense_true[:, 2]
depth2 = (R_true @ recon_dense_true.T + t_true.reshape(3, 1))[2]
valid = (depth1 > 0) & (depth2 > 0)
x1d, recon_dense_true, recon_dense_estimated = x1d[valid], recon_dense_true[valid], recon_dense_estimated[valid]

rgb1 = cv2.cvtColor(view1, cv2.COLOR_BGR2RGB)
px = np.clip(x1d[:, 0].astype(int), 0, rgb1.shape[1] - 1)
py = np.clip(x1d[:, 1].astype(int), 0, rgb1.shape[0] - 1)
point_colors = rgb1[py, px] / 255.0

print(f'{len(x1d)} points in the denser cloud (vs. {len(x1_in)} used for the numeric check above)')

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*recon_dense_true.T, c='gray', s=8, alpha=0.3, label='triangulated with true poses')
ax.scatter(*recon_dense_estimated.T, c=point_colors, s=25, label='reconstructed (estimated pose)')

# mark the two camera centers
cam1_center = -R1.T @ t1
cam2_center = -R_true.T @ t_true
ax.scatter(*cam1_center, c='blue', s=80, marker='^', label='camera 1')
ax.scatter(*cam2_center, c='green', s=80, marker='^', label='camera 2')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.legend(fontsize=8)
ax.set_title('Structure from motion: recovered 3D points and camera poses')
plt.show()

### Od rijetkog do gustog oblaka

Čak i uz prethodno ublažene pragove, podudaranje značajki daje samo *rijedak* oblak &mdash; jednu 3D točku po prepoznatljivoj ključnoj točki slike. Kao što je prikazano gore, rijedak oblak često nije osobito zadovoljavajući. No uz sada poznate položaje i orijentacije kamera više nismo ograničeni na prepoznatljive ključne točke: **stereo iz više pogleda (MVS)** procjenjuje dubinu za *svaki* piksel u *svakom* pogledu, pomicanjem hipoteze ravnine ili lokalnog područja kroz prostor (poopćenje stereo podudaranja iz lekcije 22 s rektificiranog para na kalibrirane kamere proizvoljnih položaja i orijentacija). Te karte dubine pojedinih pogleda zatim se spajaju &mdash; nedosljedne procjene se filtriraju, a ostale sjedinjuju &mdash; u gusti obojeni oblak točaka (ili mrežu) koji stvaraju alati poput COLMAP-a (<a href="../references.html#schonberger-2016-colmap">Schönberger &amp; Frahm, 2016.</a>).

### Od dvaju pogleda do mnogih: inkrementalni SfM i zajednička optimizacija

Stvarne rekonstrukcije rijetko završavaju na dvama pogledima. Standardni postupak redom uključuje nove slike u 3 koraka: 1) podudariti značajke između nove slike i jedne ili više postojećih slika; 2) izračunati položaj i orijentaciju nove kamere u odnosu na postojeći 3D oblak točaka; 3) ažurirati 3D oblak točaka novim podudaranjima pomoću triangulacije. Korak 2 zahtijeva rješavanje problema Perspective-n-Point (odnosno, uz 2D–3D korespondencije i $K$, odrediti položaj i orijentaciju te kamere pomoću `cv2.solvePnP`).

Kako se pogledi prikupljaju, pogreške se gomilaju &mdash; položaj i orijentacija kvare triangulirane točke, koje zatim kvare sljedeću procjenu položaja i orijentacije, i tako dalje. **Zajednička optimizacija kamera i točaka (bundle adjustment)** to ispravlja usavršavanjem svih položaja i orijentacija kamera i svih 3D točaka *zajedno*, u jednoj velikoj nelinearnoj optimizaciji najmanjih kvadrata koja minimizira ukupnu pogrešku reprojekcije zbrojenu preko svih točaka u svim pogledima odjednom. Može se reći da je to najvažniji pojedinačni korak koji razlikuje jednostavan postupak s dvama pogledima poput ovog od stvarnog SfM sustava (COLMAP i dr.).

### Zadaci

1. Prije triangulacije dodajte dodatni sintetički pikselni šum (npr. `rng.normal(0, 1.0, x1_in.shape)`) povrh već stvarnih detekcija u `x1_in`/`x2_in`. Koliko raste pogreška rekonstrukcije i raste li ravnomjerno ili više za točke udaljenije od kamera (odnos dispariteta i dubine iz lekcije 22: udaljene točke daju manju, bučniju paralaksu)?
2. Pronađite potpis funkcije `cv2.solvePnP` i opišite riječima kako biste proširili ovaj notebook na treći pogled: koje biste 3D točke iz rekonstrukcije `view1`/`view2` podudarili s novom slikom i kako bi se ulazi i izlazi funkcije `solvePnP` povezali s dijelovima koje već imamo (`K`, podudarene 2D točke i postojeći 3D oblak točaka)?
3. Ovaj par (pogledi 0 i 2) namjerno ima velik razmak kamera za pouzdano podudaranje. Umjesto njega koristite poglede 0 i 1 (mnogo manji razmak &mdash; trebat će vam njihovi parametri kamera u `.npz` datotekama iz istog prizora BlendedMVS-a). Kako manji razmak utječe na broj točaka koje prolaze RANSAC te na točnost procijenjenog položaja i orijentacije i rekonstrukcije u usporedbi s gore korištenim parom većeg razmaka?